# Qwen3Guard-Gen-4B + Presidio PII Detection Test

Runs the PII detection evaluation against **Qwen/Qwen3Guard-Gen-4B** using transformers with 4-bit quantization, combined with **Microsoft Presidio** for improved recall.

**Modes tested:**
- **Qwen3Guard only** (section 5) — model-based PII detection
- **Combined** (section 7) — Presidio + Qwen3Guard (logical OR — PII detected if either flags it)

**Requirements:** GPU runtime (T4 or better). Go to **Runtime > Change runtime type > T4 GPU**.

**VRAM usage:** ~5-6GB with 4-bit quantization (fits comfortably on T4's 16GB).

## 1. Install Dependencies

In [ ]:
!pip install -U transformers accelerate bitsandbytes tabulate tqdm presidio-analyzer presidio-anonymizer spacy
!python -m spacy download en_core_web_lg
!python -m spacy download xx_ent_wiki_sm

## 2. Clone Repo

In [ ]:
!git clone https://github.com/FuseableTechnologies/ollama-qwen3guard-test.git
%cd ollama-qwen3guard-test

## 3. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 4. Quick Sanity Check

Load the model with 4-bit quantization and test a single query.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen3Guard-Gen-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype="float16",
    ),
)
print("Model loaded!")

In [ ]:
query = "Client John Smith, SSN 123-45-6789, wants to open an account."
messages = [{"role": "user", "content": query}]

text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer([text], return_tensors="pt").to(model.device)
generated = model.generate(**inputs, max_new_tokens=128)
output = tokenizer.decode(generated[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

print(output)

In [ ]:
# Free the model before running the full eval (it will reload)
del model, tokenizer
import torch; torch.cuda.empty_cache()

## 5. Run Full Evaluation

In [ ]:
!mkdir -p results
!python detect_pii.py \
    --local \
    --4bit \
    --model Qwen/Qwen3Guard-Gen-4B \
    --output results/qwen3guard_gen_4b_results.json \
    --verbose

## 6. View Results

In [ ]:
import json

with open("results/qwen3guard_gen_4b_results.json") as f:
    data = json.load(f)

print(f"Model: {data['model']}")
print(f"Total: {data['total_entries']}")
for k, v in data["metrics"].items():
    if k != "confusion_matrix":
        print(f"{k}: {v:.3f}")
print(f"Confusion matrix: {data['metrics']['confusion_matrix']}")

## 7. Run Combined (Presidio + Qwen3Guard) Evaluation

Runs both detectors and flags PII if **either** detects it (logical OR).

In [ ]:
!python detect_pii.py \
    --combined \
    --local \
    --4bit \
    --model Qwen/Qwen3Guard-Gen-4B \
    --output results/combined_results.json \
    --verbose

## 8. Compare Qwen3Guard vs Combined

In [ ]:
import json
from tabulate import tabulate

files = {
    "Qwen3Guard": "results/qwen3guard_gen_4b_results.json",
    "Combined": "results/combined_results.json",
}

rows = []
for name, path in files.items():
    try:
        with open(path) as f:
            data = json.load(f)
        m = data["metrics"]
        cm = m["confusion_matrix"]
        rows.append([
            name,
            f"{m['accuracy']:.3f}",
            f"{m['precision']:.3f}",
            f"{m['recall']:.3f}",
            f"{m['f1']:.3f}",
            f"TP={cm['tp']} FP={cm['fp']} FN={cm['fn']} TN={cm['tn']}",
        ])
    except FileNotFoundError:
        rows.append([name, "—", "—", "—", "—", "results file not found"])

print(tabulate(rows, headers=["Mode", "Acc", "Prec", "Recall", "F1", "Confusion"], tablefmt="grid"))